# Step 6 — Rolling walk-forward evaluation

Steps 1–5 used a **single static train/test split** (train on `t ≤ 34`, predict all `t ≥ 35` in one shot). This step replaces that with **rolling walk-forward**: at each test timestep `t*`, retrain the model on **all labelled data with `t ≤ t* − 1`**, then predict only the txs at exactly `t = t*`.

## Why this matters

1. **Realistic deployment**: in a fraud-detection system, labels arrive over time. By the time you classify txs at `t = 42`, you have ground-truth labels through `t = 41`. The static split throws this away.
2. **Directly attacks the t ≥ 43 cliff**: by the time we predict at `t = 43`, the model has been trained on labels through `t = 42` — including the dark-market-shutdown signal at `t = 42`. The static-split model never sees post-shutdown training data.
3. **Closer-in-distribution validation**: at each `t*`, the val window is `t ∈ [t* − 5, t* − 1]` rather than the fixed `[30, 34]` (which is in-distribution with train but out-of-distribution with the post-cliff test).

## Causality contract — unchanged

Features are already causal: each tx at time `t` uses only `τ < t` priors (trajectory + Block B + PPR). They were computed *once* over the full dataset using ground-truth labels for every tx, but each tx's feature only ever uses labels of txs at strictly earlier timesteps. So the same feature matrix is reusable here — only the *training mask* changes per `t*`.

## Models swept

1. **RF C1** — Random Forest on the 233-dim C1 feature set, retrained at each `t*`.
2. **LightGBM C1** — early-stopped on val window `[t* − 5, t* − 1]`, retrained at each `t*`.
3. **Ensemble** — RF + LGB soft-vote with weights tuned on the per-step val window.

Compared to the static-split versions:
- Step 4 RF C1 static: F1 = 0.8149
- Step 5 LightGBM C1 static (F1@0.5): F1 = 0.8004
- Step 5 ensemble static (F1@val-thr): F1 = 0.8118

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, classification_report, precision_recall_curve
import lightgbm as lgb

ROOT = Path.cwd()
while not (ROOT / "transactions_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
WALLETS_DIR      = ROOT / "actors_data"
CACHE_DIR        = ROOT / "architectures" / "inductive_tx_classification" / "cross_step_tx_graph" / "cache"
print(f"repo root: {ROOT}")

TRAIN_END    = 34
N_TIME_STEPS = 49
RANDOM_SEED  = 175
VAL_WIN      = 5    # val window: [t*-VAL_WIN, t*-1]
np.random.seed(RANDOM_SEED)

repo root: /Users/bryanfang/Library/CloudStorage/OneDrive-HarvardUniversity/School/2025-2026/STAT 175/175_final_project


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/dask/dataframe/__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
print("Loading raw data + cached features...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.txt")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.txt")
tx_classes_df["label"] = tx_classes_df["class"].map({1:1,2:0,3:-1}).astype(np.int8)
all_cols = list(tx_features_df.columns)
GRAPH_COLS = [c for c in all_cols if c.startswith("Aggregate_feature")] + ["in_txs_degree","out_txs_degree"]
feat_cols_intr = [c for c in all_cols if c not in ("txId","Time step") and c not in GRAPH_COLS]
F_INTRINSIC = len(feat_cols_intr)
merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
tx_X_intr = tx_features_df[feat_cols_intr].fillna(0).astype(np.float32).values

# Load cached feature blocks
traj_X    = np.load(CACHE_DIR / "traj_X.npy")
block_B_X = np.load(CACHE_DIR / "block_B_X.npy")
block_C_X = np.load(CACHE_DIR / "block_C_X.npy")
X_C1 = np.concatenate([tx_X_intr, traj_X, block_B_X, block_C_X], axis=1)
print(f"  X_C1: {X_C1.shape}")
print(f"  N_tx: {len(tx_time):,}  illicit: {(tx_label==1).sum():,}  licit: {(tx_label==0).sum():,}")

Loading raw data + cached features...


  X_C1: (203769, 233)
  N_tx: 203,769  illicit: 4,545  licit: 42,019


In [3]:
# Recompute the static-split RF C1 number for comparison.
labelled = (tx_label != -1)
stat_train_mask = labelled & (tx_time <= TRAIN_END)
stat_test_mask  = labelled & (tx_time >  TRAIN_END)
y_test_stat = tx_label[stat_test_mask]
test_t_stat = tx_time[stat_test_mask]

rf_static = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                    n_jobs=-1, random_state=RANDOM_SEED)
rf_static.fit(X_C1[stat_train_mask], tx_label[stat_train_mask])
p_stat = rf_static.predict_proba(X_C1[stat_test_mask])[:,1]
yp_stat = (p_stat >= 0.5).astype(int)
f1_stat = f1_score(y_test_stat, yp_stat, pos_label=1, zero_division=0)
auc_stat = roc_auc_score(y_test_stat, p_stat)
pr_stat  = average_precision_score(y_test_stat, p_stat)
print(f"=== Static-split RF C1 (reference) ===")
print(f"  F1={f1_stat:.4f}  AUC={auc_stat:.4f}  PR-AUC={pr_stat:.4f}")

=== Static-split RF C1 (reference) ===
  F1=0.8146  AUC=0.9209  PR-AUC=0.8067


In [4]:
# Rolling walk-forward with RF C1.
# At each t* in [35, 49]:
#   train_mask = labelled & (tx_time <= t* - 1)
#   test_mask  = labelled & (tx_time == t*)
#   train RF, predict test, store probs.

print("=== Walk-forward RF C1 ===")
wf_p_rf   = np.full(len(tx_time), np.nan, dtype=np.float64)
wf_yp_rf  = np.full(len(tx_time), -1, dtype=np.int64)

t0 = time.time()
for tstar in range(TRAIN_END+1, N_TIME_STEPS+1):
    tr_mask = labelled & (tx_time <= tstar - 1)
    te_mask = labelled & (tx_time == tstar)
    n_tr = int(tr_mask.sum()); n_te = int(te_mask.sum())
    if n_te == 0 or tx_label[te_mask].sum() == 0 and tx_label[te_mask].size == 0:
        continue
    rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                 n_jobs=-1, random_state=RANDOM_SEED)
    rf.fit(X_C1[tr_mask], tx_label[tr_mask])
    p_te = rf.predict_proba(X_C1[te_mask])[:,1]
    yp_te = (p_te >= 0.5).astype(int)
    wf_p_rf[te_mask]  = p_te
    wf_yp_rf[te_mask] = yp_te
    n_pos = int(tx_label[te_mask].sum())
    if n_pos > 0:
        f1t = f1_score(tx_label[te_mask], yp_te, pos_label=1, zero_division=0)
    else:
        f1t = float('nan')
    print(f"  t*={tstar:>2d}  n_train={n_tr:>5,}  n_test={n_te:>4,}  illicit={n_pos:>3d}  F1={f1t:.4f}")
print(f"  total walk-forward RF time: {time.time()-t0:.0f}s")

# Aggregate over the test horizon (t >= 35)
test_mask = labelled & (tx_time > TRAIN_END)
y_test = tx_label[test_mask]
p_wf_rf  = wf_p_rf[test_mask]
yp_wf_rf = wf_yp_rf[test_mask]
f1_wf_rf = f1_score(y_test, yp_wf_rf, pos_label=1, zero_division=0)
auc_wf_rf = roc_auc_score(y_test, p_wf_rf)
pr_wf_rf  = average_precision_score(y_test, p_wf_rf)
print(f"\n  Aggregate (test t in [35,49]):  F1={f1_wf_rf:.4f}  AUC={auc_wf_rf:.4f}  PR-AUC={pr_wf_rf:.4f}")

=== Walk-forward RF C1 ===


  t*=35  n_train=29,894  n_test=1,341  illicit=182  F1=0.9663


  t*=36  n_train=31,235  n_test=1,708  illicit= 33  F1=0.9552


  t*=37  n_train=32,943  n_test= 498  illicit= 40  F1=0.7692


  t*=38  n_train=33,441  n_test= 756  illicit=111  F1=0.9223


...

  t*=45  n_train=42,838  n_test=1,221  illicit=  5  F1=0.0000


  t*=46  n_train=44,059  n_test= 712  illicit=  2  F1=0.6667


  t*=47  n_train=44,771  n_test= 846  illicit= 22  F1=0.0000


  t*=48  n_train=45,617  n_test= 471  illicit= 36  F1=0.0541


  t*=49  n_train=46,088  n_test= 476  illicit= 56  F1=0.4658
  total walk-forward RF time: 51s

  Aggregate (test t in [35,49]):  F1=0.8240  AUC=0.9721  PR-AUC=0.8762


In [5]:
# Rolling walk-forward with LightGBM. At each t*, val window = [t*-VAL_WIN, t*-1] for early stopping.
# Final fit on full train (<=t*-1) at val-selected n_estimators.

print("=== Walk-forward LightGBM C1 ===")
wf_p_lgb  = np.full(len(tx_time), np.nan, dtype=np.float64)
wf_yp_lgb = np.full(len(tx_time), -1, dtype=np.int64)
lgb_iters = {}
lgb_val_thr = {}

t0 = time.time()
for tstar in range(TRAIN_END+1, N_TIME_STEPS+1):
    tr_es_mask = labelled & (tx_time <= tstar - VAL_WIN - 1)        # for early stop fit
    val_mask   = labelled & (tx_time >= tstar - VAL_WIN) & (tx_time <= tstar - 1)
    full_mask  = labelled & (tx_time <= tstar - 1)
    te_mask    = labelled & (tx_time == tstar)
    if int(te_mask.sum()) == 0:
        continue
    if int(val_mask.sum()) < 50 or int(tr_es_mask.sum()) < 100:
        # not enough data — fall back to plain fit
        n_iter = 200
    else:
        y_tr_es = tx_label[tr_es_mask]; y_val = tx_label[val_mask]
        pos_w_es = (1.0-y_tr_es.mean())/max(y_tr_es.mean(), 1e-6)
        m_es = lgb.LGBMClassifier(
            objective="binary", n_estimators=2000,
            learning_rate=0.1, num_leaves=63, min_child_samples=20,
            subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
            scale_pos_weight=pos_w_es, verbose=-1, n_jobs=-1, random_state=RANDOM_SEED,
        )
        m_es.fit(X_C1[tr_es_mask], y_tr_es,
                 eval_set=[(X_C1[val_mask], y_val)],
                 callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)])
        n_iter = max(int(m_es.best_iteration_ or 200), 50)
        # threshold tune on val
        p_val = m_es.predict_proba(X_C1[val_mask])[:,1]
        prec, rec, thr = precision_recall_curve(y_val, p_val)
        f1s = 2*prec*rec/(prec+rec+1e-12)
        lgb_val_thr[tstar] = float(thr[int(np.argmax(f1s[:-1]))]) if len(thr)>0 else 0.5
    lgb_iters[tstar] = n_iter
    y_full = tx_label[full_mask]
    pos_w_full = (1.0-y_full.mean())/max(y_full.mean(), 1e-6)
    m_full = lgb.LGBMClassifier(
        objective="binary", n_estimators=n_iter,
        learning_rate=0.1, num_leaves=63, min_child_samples=20,
        subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
        scale_pos_weight=pos_w_full, verbose=-1, n_jobs=-1, random_state=RANDOM_SEED,
    )
    m_full.fit(X_C1[full_mask], y_full)
    p_te = m_full.predict_proba(X_C1[te_mask])[:,1]
    thr_use = lgb_val_thr.get(tstar, 0.5)
    yp_te_05  = (p_te >= 0.5).astype(int)
    yp_te_thr = (p_te >= thr_use).astype(int)
    wf_p_lgb[te_mask]  = p_te
    wf_yp_lgb[te_mask] = yp_te_05      # report F1@0.5; threshold variant computed separately below
    n_pos = int(tx_label[te_mask].sum())
    if n_pos > 0:
        f1_05 = f1_score(tx_label[te_mask], yp_te_05, pos_label=1, zero_division=0)
        f1_th = f1_score(tx_label[te_mask], yp_te_thr, pos_label=1, zero_division=0)
    else: f1_05=float('nan'); f1_th=float('nan')
    print(f"  t*={tstar:>2d}  n_full={int(full_mask.sum()):>5,}  n_te={int(te_mask.sum()):>4,}  ill={n_pos:>3d}  "
          f"iters={n_iter:>3d}  thr={thr_use:.3f}  F1@0.5={f1_05:.4f}  F1@thr={f1_th:.4f}")
print(f"  total walk-forward LGB time: {time.time()-t0:.0f}s")

# Aggregate at threshold 0.5
p_wf_lgb = wf_p_lgb[test_mask]
yp_wf_lgb_05 = (p_wf_lgb >= 0.5).astype(int)
# Aggregate at per-step val-tuned threshold
yp_wf_lgb_thr = np.zeros_like(y_test)
for tstar in range(TRAIN_END+1, N_TIME_STEPS+1):
    m_t = (tx_time[test_mask] == tstar)
    if not m_t.any(): continue
    yp_wf_lgb_thr[m_t] = (p_wf_lgb[m_t] >= lgb_val_thr.get(tstar, 0.5)).astype(int)
f1_wf_lgb_05 = f1_score(y_test, yp_wf_lgb_05, zero_division=0)
f1_wf_lgb_thr = f1_score(y_test, yp_wf_lgb_thr, zero_division=0)
auc_wf_lgb = roc_auc_score(y_test, p_wf_lgb)
pr_wf_lgb  = average_precision_score(y_test, p_wf_lgb)
print(f"\n  Aggregate (test t in [35,49]):")
print(f"    F1@0.5={f1_wf_lgb_05:.4f}  F1@val-thr={f1_wf_lgb_thr:.4f}  AUC={auc_wf_lgb:.4f}  PR-AUC={pr_wf_lgb:.4f}")

=== Walk-forward LightGBM C1 ===


  t*=35  n_full=29,894  n_te=1,341  ill=182  iters=179  thr=0.708  F1@0.5=0.9725  F1@thr=0.9640


  t*=36  n_full=31,235  n_te=1,708  ill= 33  iters=161  thr=0.840  F1@0.5=0.9706  F1@thr=0.9697


  t*=37  n_full=32,943  n_te= 498  ill= 40  iters=172  thr=0.933  F1@0.5=0.7879  F1@thr=0.7692


  t*=38  n_full=33,441  n_te= 756  ill=111  iters= 95  thr=0.630  F1@0.5=0.9378  F1@thr=0.9327


...

  t*=45  n_full=42,838  n_te=1,221  ill=  5  iters= 59  thr=0.612  F1@0.5=0.0000  F1@thr=0.0000


  t*=46  n_full=44,059  n_te= 712  ill=  2  iters=109  thr=0.861  F1@0.5=0.6667  F1@thr=0.6667


  t*=47  n_full=44,771  n_te= 846  ill= 22  iters= 74  thr=0.769  F1@0.5=0.0000  F1@thr=0.0000


  t*=48  n_full=45,617  n_te= 471  ill= 36  iters=107  thr=0.211  F1@0.5=0.4167  F1@thr=0.5517


  t*=49  n_full=46,088  n_te= 476  ill= 56  iters=120  thr=0.069  F1@0.5=0.9636  F1@thr=0.8871
  total walk-forward LGB time: 77s

  Aggregate (test t in [35,49]):
    F1@0.5=0.8557  F1@val-thr=0.8405  AUC=0.9749  PR-AUC=0.8988


In [6]:
# Ensemble RF and LGB walk-forward predictions, weights chosen by per-step val PR-AUC.
# Quick approach: equal weight 0.5/0.5 (averages stabilize across both learners).
p_wf_ens = 0.5*p_wf_rf + 0.5*p_wf_lgb
yp_wf_ens_05 = (p_wf_ens >= 0.5).astype(int)
f1_ens_05 = f1_score(y_test, yp_wf_ens_05, zero_division=0)
auc_ens = roc_auc_score(y_test, p_wf_ens); pr_ens = average_precision_score(y_test, p_wf_ens)
print(f"=== Ensemble walk-forward (RF + LGB, equal weights, thr=0.5) ===")
print(f"  F1={f1_ens_05:.4f}  AUC={auc_ens:.4f}  PR-AUC={pr_ens:.4f}")

=== Ensemble walk-forward (RF + LGB, equal weights, thr=0.5) ===
  F1=0.8532  AUC=0.9780  PR-AUC=0.9001


In [7]:
print("="*78)
print("SUMMARY — STATIC vs WALK-FORWARD")
print("="*78)
print(f"{'Setup':40s}  {'F1':>7s}  {'AUC':>7s}  {'PR-AUC':>7s}")
print("-"*78)
rows = [
    ("STATIC: RF C1 [233] (step 4)",          f1_stat,       auc_stat,      pr_stat),
    ("WALK-FWD: RF C1 [233]",                 f1_wf_rf,      auc_wf_rf,     pr_wf_rf),
    ("WALK-FWD: LGB C1 @ thr=0.5",            f1_wf_lgb_05,  auc_wf_lgb,    pr_wf_lgb),
    ("WALK-FWD: LGB C1 @ per-step val-thr",   f1_wf_lgb_thr, auc_wf_lgb,    pr_wf_lgb),
    ("WALK-FWD: RF+LGB equal-weight ensemble",f1_ens_05,     auc_ens,       pr_ens),
]
for n, f1, auc, pr in rows:
    print(f"  {n:40s}  {f1:>7.4f}  {auc:>7.4f}  {pr:>7.4f}")

# Per-timestep comparison: static RF vs walk-forward RF vs walk-forward LGB+thr vs ensemble
print("\n"+"="*78)
print("Per-timestep test F1 (illicit class)")
print("="*78)
print(f"{'t':>3}  {'n':>5}  {'ill':>4}  {'static_RF':>10}  {'wf_RF':>10}  {'wf_LGB05':>10}  {'wf_LGBthr':>10}  {'wf_ens':>10}")
yp_stat_arr = (p_stat >= 0.5).astype(int)
for tstar in range(TRAIN_END+1, N_TIME_STEPS+1):
    m = (test_t_stat == tstar)
    if not m.any(): continue
    yt = y_test_stat[m]
    if yt.sum() == 0:
        f_st = f_wf = f_lg = f_lgt = f_ens = float('nan')
    else:
        f_st = f1_score(yt, yp_stat_arr[m], pos_label=1, zero_division=0)
        f_wf = f1_score(yt, yp_wf_rf[m], pos_label=1, zero_division=0)
        f_lg = f1_score(yt, yp_wf_lgb_05[m], pos_label=1, zero_division=0)
        f_lgt= f1_score(yt, yp_wf_lgb_thr[m], pos_label=1, zero_division=0)
        f_ens= f1_score(yt, yp_wf_ens_05[m], pos_label=1, zero_division=0)
    print(f"{tstar:>3}  {int(m.sum()):>5}  {int(yt.sum()):>4}  "
          f"{f_st:>10.4f}  {f_wf:>10.4f}  {f_lg:>10.4f}  {f_lgt:>10.4f}  {f_ens:>10.4f}")

# Cliff-region focus
print("\nFocus: t >= 43 (the cliff)")
cliff = test_t_stat >= 43
if cliff.any() and y_test_stat[cliff].sum() > 0:
    print(f"  static RF       F1={f1_score(y_test_stat[cliff], yp_stat_arr[cliff], zero_division=0):.4f}  "
          f"n={int(cliff.sum())} ill={int(y_test_stat[cliff].sum())}")
    print(f"  walk-fwd RF     F1={f1_score(y_test_stat[cliff], yp_wf_rf[cliff], zero_division=0):.4f}")
    print(f"  walk-fwd LGB05  F1={f1_score(y_test_stat[cliff], yp_wf_lgb_05[cliff], zero_division=0):.4f}")
    print(f"  walk-fwd LGBthr F1={f1_score(y_test_stat[cliff], yp_wf_lgb_thr[cliff], zero_division=0):.4f}")
    print(f"  walk-fwd ens    F1={f1_score(y_test_stat[cliff], yp_wf_ens_05[cliff], zero_division=0):.4f}")

SUMMARY — STATIC vs WALK-FORWARD
Setup                                          F1      AUC   PR-AUC
------------------------------------------------------------------------------
  STATIC: RF C1 [233] (step 4)               0.8146   0.9209   0.8067
  WALK-FWD: RF C1 [233]                      0.8240   0.9721   0.8762
  WALK-FWD: LGB C1 @ thr=0.5                 0.8557   0.9749   0.8988
  WALK-FWD: LGB C1 @ per-step val-thr        0.8405   0.9749   0.8988
  WALK-FWD: RF+LGB equal-weight ensemble     0.8532   0.9780   0.9001

Per-timestep test F1 (illicit class)
  t      n   ill   static_RF       wf_RF    wf_LGB05   wf_LGBthr      wf_ens
 35   1341   182      0.9663      0.9663      0.9725      0.9640      0.9640
 36   1708    33      0.9429      0.9552      0.9706      0.9697      0.9851
 37    498    40      0.7879      0.7692      0.7879      0.7692      0.7879
 38    756   111      0.9275      0.9223      0.9378      0.9327      0.9327
 39   1183    81      0.9231      0.8980      0